# DiagnoTech Model 2 · Notebook 02 — DenseNet121 Training, Optimisation & Resumption

**Scope.** Staged DenseNet121 training for the eight-output multi-label classifier:

* **Phase A (feature extraction).** Backbone frozen, head trained at LR 1e-3, batch 16, up to 10 epochs.
* **Phase B (fine-tuning).** Stage 1 unfreezes `denseblock4` and `norm5`. Stage 2 adds `transition3` and `denseblock3`. Differential LR: head 5e-4, backbone 1e-4. Up to 30 epochs in total, with validation-driven early stopping.
* **Regularisation and imbalance.** Dropout 0.3, AdamW with weight decay 1e-4, conservative augmentation (no flips), `pos_weight` or `WeightedRandomSampler`, optional label smoothing 0.05.
* **Adaptive LR.** ReduceLROnPlateau on validation loss (factor 0.5, patience 2, min 1e-7).

**Experiments E0 to E8** are defined in `config/experiments.json`. Each run writes to its own folder, and runs are compared at the end of this notebook.

**Data discipline.** This notebook loads only the `train` and `val` manifests. The test split is untouched until notebook 03.

**Resumption.** If the Colab runtime disconnects, reconnect, re-run sections 1 to 3, and re-run section 5. Training continues from the last completed epoch. It does not restart from epoch 1.

## 1. Environment and project context

In [ ]:
import os
import sys
from pathlib import Path

CANDIDATE_CODE_ROOTS = [
    os.environ.get("DIAGNOTECH_CODE_ROOT"),
    "/content/DiagnoTech_Model2",
    str(Path.cwd()),
    str(Path.cwd().parent),
]
code_roots = [Path(c) for c in CANDIDATE_CODE_ROOTS if c and (Path(c) / "src" / "diagnotech").is_dir()]
if not code_roots:
    raise FileNotFoundError("Project code not found. Unzip DiagnoTech_Model2_Professional.zip to /content first.")
CODE_ROOT = code_roots[0]
sys.path.insert(0, str(CODE_ROOT / "src"))

import pandas as pd
import torch

from diagnotech.config import in_colab, load_context  # noqa: E402

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")

ctx = load_context(CODE_ROOT)
cfg = ctx.project
CLASSES = list(ctx.classes)
RAW_DIR = Path(os.environ.get("DIAGNOTECH_RAW_DIR", cfg["source"]["colab_raw_data_dir"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 1024**3:.1f} GB | torch {torch.__version__}")
else:
    print("WARNING: no GPU. Training will run but is impractically slow on CPU.")
print("Storage root:", ctx.storage.root)

## 2. Load training and validation manifests
Images are resolved against the raw-data directory of this session. Notebook 01 must have extracted the archive in this session, so that the `.extracted_ok` marker exists.

In [ ]:
from diagnotech.data_manifest import load_manifest, resolve_image_paths

if not (RAW_DIR / ".extracted_ok").exists():
    raise FileNotFoundError("Dataset not extracted in this session. Run notebook 01 first.")

manifests = {}
for split in ("train", "val"):  # the test split is intentionally NOT loaded in this notebook
    df = resolve_image_paths(load_manifest(ctx.storage.data / f"manifest_{split}.csv"), RAW_DIR)
    missing = int((~df["image_path"].map(os.path.exists)).sum())
    if missing:
        raise FileNotFoundError(f"{missing} {split} images are missing. Re-run notebook 01.")
    manifests[split] = df
    print(f"{split:>5}: {len(df):>7,} images | {df['patient_id'].nunique():>6,} patients")

## 3. Run configuration
Set the experiment ID. Run E0 first, then E1, and so on, because each experiment adds one documented change. The batch size may be 8, 16 or 32. Larger batches are faster but use more VRAM. Smaller batches can generalise slightly better on this kind of data. Keep the batch size fixed within an experiment.

In [ ]:
EXPERIMENT_ID = "E0"   # E0 ... E8
BATCH_SIZE = 16        # 8 | 16 | 32
NUM_WORKERS = 2        # 2 is a safe Colab default
RESUME = True          # resume from checkpoints/<ID>/last_checkpoint.pth when present

from diagnotech.training import run_experiment  # noqa: E402

assert BATCH_SIZE in cfg["training"]["allowed_batch_sizes"], "Batch size must be 8, 16 or 32."
print(f"Experiment {EXPERIMENT_ID}: {ctx.experiments[EXPERIMENT_ID]['description']}")

## 4. Train (or resume) the experiment
Each epoch logs training and validation loss, per-label accuracy, macro F1, macro AUROC, learning rate, epoch time and GPU memory. The best checkpoint is selected by validation loss. Early stopping is active only in experiments E6 to E8. Earlier experiments use fixed budgets, as the sequence requires.

In [ ]:
summary = run_experiment(
    EXPERIMENT_ID,
    ctx,
    manifests,
    device,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    resume=RESUME,
)
pd.Series(summary).to_frame("value")

## 5. Training curves (required plots)
Loss, accuracy and macro F1 are plotted against epoch for training and validation. A learning-rate curve is shown alongside. Dashed lines mark the start of each training stage.

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(ctx.storage.checkpoints / EXPERIMENT_ID / "training_history.csv")
epochs = history["global_epoch"]
stage_starts = history.loc[history["epoch_in_stage"] == 1, "global_epoch"].iloc[1:]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0, 0]
ax.plot(epochs, history["train_loss"], label="train")
ax.plot(epochs, history["val_loss"], label="validation")
ax.set_title("Training vs validation loss")
ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy")

ax = axes[0, 1]
ax.plot(epochs, history["train_label_accuracy"], label="train")
ax.plot(epochs, history["val_label_accuracy"], label="validation")
ax.set_title("Training vs validation accuracy (per label, threshold 0.5)")
ax.set_xlabel("epoch")
ax.set_ylabel("per-label accuracy")

ax = axes[1, 0]
ax.plot(epochs, history["train_macro_f1"], label="train")
ax.plot(epochs, history["val_macro_f1"], label="validation")
ax.set_title("Macro F1 vs epoch (threshold 0.5)")
ax.set_xlabel("epoch")
ax.set_ylabel("macro F1")

ax = axes[1, 1]
ax.plot(epochs, history["lr_head"], label="head")
if history["lr_backbone"].notna().any():
    ax.plot(epochs, history["lr_backbone"], label="backbone")
ax.set_yscale("log")
ax.set_title("Learning rate vs epoch")
ax.set_xlabel("epoch")
ax.set_ylabel("learning rate (log scale)")

for ax in axes.ravel():
    ax.grid(alpha=0.3)
    ax.legend()
    for boundary in stage_starts:
        ax.axvline(boundary, ls="--", color="grey", alpha=0.6)
fig.suptitle(f"{EXPERIMENT_ID} training curves")
fig.tight_layout()
fig.savefig(ctx.storage.reports / f"{EXPERIMENT_ID}_training_curves.png", bbox_inches="tight")
plt.show()

## 6. Validation per-class performance for this run
Per-class validation metrics use the best checkpoint and thresholds tuned on the validation split.

In [ ]:
val_per_class = pd.read_csv(ctx.storage.experiments / f"{EXPERIMENT_ID}_val_per_class.csv")
val_per_class.round(4)

## 7. Experiment comparison
Experiments are ranked by validation macro AUROC. Any experiment whose AUROC falls more than 0.02 below E0 on any single class is marked ineligible. This guards against a configuration that improves the average while hurting one clinically important class. Macro F1 at tuned thresholds is shown but not used for ranking, because thresholds are tuned on the same validation data.

In [ ]:
from diagnotech.metrics import rank_experiments

results = pd.read_csv(ctx.storage.experiments / "experiment_results.csv")
ranking = rank_experiments(
    results,
    baseline_id=cfg["selection"]["baseline_experiment"],
    max_class_auroc_drop=cfg["selection"]["max_class_auroc_drop_vs_baseline"],
)
show_cols = ["rank", "experiment_id", "eligible", "max_class_auroc_drop", "val_macro_auroc",
             "val_macro_f1_tuned", "val_loss", "val_label_accuracy", "best_epoch", "training_seconds"]
ranking[show_cols].round(4)

## Next step
Once the comparison supports a configuration, run notebook **03_Evaluation_Deployment.ipynb**. It reads the best checkpoint of the top-ranked experiment, or of an experiment you name, and evaluates it once on the untouched test split.